# Network Intrusion Detection Using Machine Learning
**Google Colab notebook | Binary classification: Normal vs Suspicious**

This notebook implements the project requirements: input validation, data quality checks, repeatable preprocessing, comparison of several classifiers, precision/recall/F1, false-positive/false-negative analysis, visualizations, and exportable results.

> **Important:** No dataset was included with the project brief. Upload a labelled network-flow CSV when prompted. Accuracy depends on the dataset and its labels; this notebook does not fabricate or guarantee accuracy above 90%. It reports the actual held-out test performance. A high score is meaningful only when the split and features avoid data leakage.

**Expected CSV:** one row per network flow, feature columns plus a label column such as `Label`, `label`, `class`, or `target`. The label should distinguish benign/normal traffic from attacks. Use public, synthetic, or explicitly authorized data only.

In [ ]:
# 1. Imports and reproducibility
import warnings, os, re, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.base import clone

RANDOM_STATE = 42
TEST_SIZE = 0.20
np.random.seed(RANDOM_STATE)
print("Imports ready. Random state:", RANDOM_STATE)

## 2. Upload your labelled dataset

Run this cell and choose your CSV file. If your dataset is large, Colab may need a few minutes. If you already mounted Google Drive, you can instead set `CSV_PATH` to the file path.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("No CSV file uploaded. Please upload a labelled network-flow CSV.")
CSV_PATH = csv_files[0]
df = pd.read_csv(CSV_PATH, low_memory=False)
print("Loaded:", CSV_PATH, "| Shape:", df.shape)
display(df.head())

## 3. Inspect and validate the input

The notebook checks empty data, duplicate rows, missing values, and likely target columns. If the automatic target choice is wrong, change `TARGET_COLUMN` below to the exact label column name.

In [ ]:
if df.empty:
    raise ValueError("The uploaded CSV is empty.")
if len(df.columns) < 2:
    raise ValueError("Expected at least one feature column and one label column.")

# Normalize column names only for target detection; original names are preserved.
target_candidates = ["label", "labels", "class", "target", "attack", "category", "is_attack", "outcome"]
normalized = {str(c).strip().lower(): c for c in df.columns}
found = next((normalized[c] for c in target_candidates if c in normalized), None)
print("Columns:", list(df.columns))
print("Rows:", len(df), "| Duplicate rows:", int(df.duplicated().sum()))
print("\nMissing values (top 20):")
display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_count"))
print("\nPossible label values:")
if found is not None:
    print("Detected target column:", repr(found))
    display(df[found].astype(str).value_counts(dropna=False).head(30).to_frame("count"))
else:
    print("No common target name detected; choose TARGET_COLUMN manually in the next cell.")

TARGET_COLUMN = found  # Change this, e.g. "Label", if needed
if TARGET_COLUMN is None or TARGET_COLUMN not in df.columns:
    raise ValueError("Set TARGET_COLUMN to the exact name of your dataset's label column.")
if df[TARGET_COLUMN].isna().all():
    raise ValueError("The selected target column contains no labels.")


## 4. Prepare binary labels and features

Normal/benign labels are mapped to `Normal`; all other labels are mapped to `Suspicious`. Review the printed mapping carefully. If your dataset uses an unusual spelling for benign traffic, add it to `NORMAL_LABELS`. The target is removed from the features to prevent direct target leakage.

In [ ]:
# Add aliases here if your dataset uses other names for normal traffic.
NORMAL_LABELS = {
    "normal", "benign", "0", "false", "no", "non-attack", "non attack",
    "normal traffic", "benign traffic"
}

work = df.copy()
# Strip whitespace in string labels and remove rows with missing target.
work[TARGET_COLUMN] = work[TARGET_COLUMN].apply(
    lambda x: x.strip() if isinstance(x, str) else x
)
work = work.dropna(subset=[TARGET_COLUMN]).copy()
raw_labels = work[TARGET_COLUMN].astype(str).str.strip()
normalized_labels = raw_labels.str.lower()
unique_labels = sorted(normalized_labels.unique().tolist())
print("Original label classes:", unique_labels[:100])

# Map binary labels: known normal values => 0, everything else => 1.
y = (~normalized_labels.isin(NORMAL_LABELS)).astype(int)
y.name = "target_binary"
print("\nBinary label counts (0=Normal, 1=Suspicious):")
display(y.map({0:"Normal", 1:"Suspicious"}).value_counts().to_frame("rows"))

if y.nunique() < 2:
    raise ValueError(
        "Only one binary class remains after mapping. Check the target column and NORMAL_LABELS."
    )

X = work.drop(columns=[TARGET_COLUMN]).copy()

# Drop columns that are entirely empty or constant. These add no useful training information.
all_missing = X.columns[X.isna().all()].tolist()
constant = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
drop_cols = sorted(set(all_missing + constant))
if drop_cols:
    print("Dropping all-missing/constant columns:", drop_cols)
    X = X.drop(columns=drop_cols)

# Remove common index-like columns; review this list if they are meaningful features.
index_like = [c for c in X.columns if str(c).strip().lower() in
              {"unnamed: 0", "index", "flow_id", "flowid"}]
if index_like:
    print("Dropping likely index/identifier columns:", index_like)
    X = X.drop(columns=index_like)

# Convert +/- infinity to missing; imputation is learned only on training data.
X = X.replace([np.inf, -np.inf], np.nan)
# Ensure at least one feature remains.
if X.shape[1] == 0:
    raise ValueError("No usable feature columns remain after cleanup.")

print("Prepared feature matrix:", X.shape)
print("Class balance:")
display(y.map({0:"Normal", 1:"Suspicious"}).value_counts(normalize=True).mul(100).round(2).to_frame("percent"))

## 5. Train/test split and preprocessing pipeline

The test set is held out until final evaluation. Numeric features are median-imputed and scaled; categorical features are most-frequent-imputed and one-hot encoded. The preprocessing is fitted inside each model pipeline to reduce leakage. For datasets with repeated flows from the same capture/session, a time- or group-aware split may be more appropriate than a random split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2))
])

transformers = []
if numeric_features:
    transformers.append(("num", numeric_transformer, numeric_features))
if categorical_features:
    transformers.append(("cat", categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Numeric features: {len(numeric_features)} | Categorical features: {len(categorical_features)}")
print("Train label counts:", y_train.value_counts().to_dict())
print("Test label counts:", y_test.value_counts().to_dict())

## 6. Compare multiple algorithms

Models included: Dummy baseline, Logistic Regression, Decision Tree, Random Forest, Extra Trees, and HistGradientBoosting (when the feature set is compatible). Models are compared on the same held-out test set. The Dummy model helps verify that machine-learning models beat a simple majority-class prediction.

In [ ]:
models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE, min_samples_leaf=2),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced_subsample", random_state=RANDOM_STATE,
        n_jobs=-1, min_samples_leaf=1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=250, class_weight="balanced", random_state=RANDOM_STATE,
        n_jobs=-1, min_samples_leaf=1
    ),
}

results = []
fitted_models = {}
for name, estimator in models.items():
    start = time.time()
    pipe = Pipeline(steps=[("preprocess", clone(preprocessor)), ("model", estimator)])
    try:
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        row = {
            "Algorithm": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "F1-score": f1_score(y_test, pred, zero_division=0),
            "Fit+predict seconds": round(time.time() - start, 2),
            "Status": "OK"
        }
        results.append(row)
        fitted_models[name] = pipe
        print(f"{name}: accuracy={row['Accuracy']:.4f}, F1={row['F1-score']:.4f}")
    except Exception as e:
        results.append({
            "Algorithm": name, "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
            "F1-score": np.nan, "Fit+predict seconds": round(time.time()-start, 2),
            "Status": f"Failed: {type(e).__name__}: {str(e)[:180]}"
        })
        print("Model failed:", name, "|", repr(e))

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False, na_position="last")
display(results_df.style.format({
    "Accuracy":"{:.2%}", "Precision":"{:.2%}", "Recall":"{:.2%}", "F1-score":"{:.2%}"
}))


## 7. Visualize model accuracy and F1-score

In [ ]:
valid_results = results_df.dropna(subset=["Accuracy"]).copy()
if not valid_results.empty:
    ax = valid_results.set_index("Algorithm")[["Accuracy", "F1-score"]].plot(
        kind="bar", figsize=(11, 5), ylim=(0, 1), rot=25
    )
    ax.set_ylabel("Score")
    ax.set_title("Held-out test performance by algorithm")
    ax.axhline(0.90, linestyle="--", linewidth=1, label="90% reference")
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()
else:
    print("No model completed successfully. Review errors in the results table.")

## 8. Evaluate the best model in detail

The model is selected by **F1-score**, not accuracy alone, because intrusion detection can have imbalanced labels. False positives are normal flows flagged as suspicious; false negatives are suspicious flows missed as normal.

In [ ]:
if not fitted_models:
    raise RuntimeError("No model fitted successfully; cannot continue evaluation.")

best_name = results_df.dropna(subset=["F1-score"]).iloc[0]["Algorithm"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_pred, labels=[0, 1])

print("Best model by test F1-score:", best_name)
print("\nClassification report (0=Normal, 1=Suspicious):")
print(classification_report(y_test, best_pred, labels=[0,1],
                            target_names=["Normal", "Suspicious"], zero_division=0))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Suspicious"])
disp.plot(values_format="d")
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives (normal correctly identified): {tn:,}")
print(f"False positives (normal flagged as suspicious): {fp:,}")
print(f"False negatives (suspicious missed): {fn:,}")
print(f"True positives (suspicious correctly detected): {tp:,}")
if accuracy_score(y_test, best_pred) >= 0.90:
    print("Accuracy reached 90% or more on this test split. This is not a guarantee of real-world performance.")
else:
    print("Accuracy is below 90% on this test split. Do not alter the test labels or report a fabricated score.")


## 9. Optional: tune Random Forest using cross-validation

This uses only the training set for model selection. It may take longer on large datasets. The final test set remains held out. Use the tuned model only if you rerun its evaluation on `X_test` after tuning.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# Optional tuning: set RUN_TUNING = True to run.
RUN_TUNING = False
if RUN_TUNING:
    rf_pipe = Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("model", RandomForestClassifier(
            class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
    param_dist = {
        "model__n_estimators": [150, 250, 400],
        "model__max_depth": [None, 15, 25, 40],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.7]
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        rf_pipe, param_distributions=param_dist, n_iter=12,
        scoring="f1", cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
    )
    search.fit(X_train, y_train)
    tuned_model = search.best_estimator_
    tuned_pred = tuned_model.predict(X_test)
    print("Best parameters:", search.best_params_)
    print("Held-out accuracy:", f"{accuracy_score(y_test, tuned_pred):.2%}")
    print("Held-out precision:", f"{precision_score(y_test, tuned_pred, zero_division=0):.2%}")
    print("Held-out recall:", f"{recall_score(y_test, tuned_pred, zero_division=0):.2%}")
    print("Held-out F1:", f"{f1_score(y_test, tuned_pred, zero_division=0):.2%}")
else:
    print("Tuning skipped. Set RUN_TUNING = True to enable.")

## 10. Export results and save the selected model

The CSV contains actual measured test scores. The model bundle includes the fitted preprocessing and classifier. Keep track of the dataset source, license, date, label definitions, and split strategy in your project report.

In [ ]:
import joblib
from google.colab import files

results_df.to_csv("algorithm_comparison_results.csv", index=False)
joblib.dump(best_model, "network_intrusion_best_model.joblib")

print("Saved: algorithm_comparison_results.csv")
print("Saved: network_intrusion_best_model.joblib")
files.download("algorithm_comparison_results.csv")
files.download("network_intrusion_best_model.joblib")

## 11. Eight documented checks / tests

Run this cell to verify basic project behavior and record the outcomes. These checks are not a replacement for testing on multiple datasets or time-separated traffic.

In [ ]:
checks = []
def record_check(name, condition, detail=""):
    checks.append({"Check": name, "Result": "PASS" if bool(condition) else "FAIL", "Detail": detail})

record_check("Input is non-empty", len(df) > 0, f"{len(df)} rows")
record_check("At least one feature", X.shape[1] > 0, f"{X.shape[1]} features")
record_check("Target column exists", TARGET_COLUMN in df.columns, str(TARGET_COLUMN))
record_check("Two binary classes", y.nunique() == 2, f"{y.nunique()} classes")
record_check("Train/test split non-empty", len(X_train) > 0 and len(X_test) > 0,
             f"train={len(X_train)}, test={len(X_test)}")
record_check("No target column in features", TARGET_COLUMN not in X.columns)
record_check("Preprocessing handles missing values", True, "Imputers are inside pipeline")
record_check("Held-out predictions produced", len(best_pred) == len(y_test), f"{len(best_pred)} predictions")
record_check("Comparison results generated", len(results_df) >= 1, f"{len(results_df)} model rows")
record_check("Metrics are within [0,1]", results_df[["Accuracy","Precision","Recall","F1-score"]]
             .dropna().apply(lambda col: col.between(0,1).all()).all())
checks_df = pd.DataFrame(checks)
display(checks_df)
if (checks_df["Result"] == "FAIL").any():
    print("One or more checks failed; review the relevant preprocessing or dataset settings.")
else:
    print(f"All {len(checks_df)} checks passed.")

## 12. Project report notes

- **Objective:** classify labelled network-flow records as normal or suspicious.
- **Methods:** compare Logistic Regression, Decision Tree, Random Forest, Extra Trees, and a Dummy baseline.
- **Evaluation:** held-out test accuracy, precision, recall, F1-score, and confusion matrix.
- **False positives:** benign traffic predicted as suspicious; can create alert fatigue.
- **False negatives:** attacks predicted as normal; can leave threats undetected.
- **Limitations:** results depend on dataset quality, label mapping, class balance, feature leakage, and whether train/test flows resemble future traffic. Random splitting may overestimate performance if near-duplicate flows or the same sessions occur in both sets.
- **Responsible use:** this is a student prototype, not a production security control. Validate on separate captures and use only authorized data.
- **Dataset documentation:** add the exact dataset name, download URL, license, collection period, and label meanings before submitting the report.